# 04. 모델 성능 비교

Logistic Regression, XGBoost, LightGBM 성능 비교 분석

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

from src.config import RANDOM_STATE, TEST_SIZE
from src.data.loader import load_gaming_behavior
from src.features.engineer import engineer_gaming_behavior_features, get_feature_columns
from src.models.trainer import get_baseline_model, get_xgboost_model, get_lgbm_model
from src.models.evaluator import evaluate_model

## 1. 데이터 준비

In [ ]:
df = load_gaming_behavior()
df = engineer_gaming_behavior_features(df)

feature_cols = get_feature_columns("kaggle")
cat_features = feature_cols["categorical"]
numeric_features = feature_cols["numeric"]

for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

all_features = numeric_features + cat_features
X = df[all_features]
y = df["is_churned"]

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=RANDOM_STATE, stratify=y_tv
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## 2. 모델 학습

In [ ]:
models = {
    "LogisticRegression": get_baseline_model(),
    "XGBoost": get_xgboost_model(n_estimators=100),
    "LightGBM": get_lgbm_model(n_estimators=100),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_proba)
    results[name] = {"model": model, "y_pred": y_pred, "y_proba": y_proba, "metrics": metrics}
    print(f"{name}: AUC-ROC={metrics['auc_roc']:.4f}, F1={metrics['f1']:.4f}")

## 3. 성능 비교 테이블

In [ ]:
comparison = []
for name, r in results.items():
    comparison.append({"Model": name, **r["metrics"]})
pd.DataFrame(comparison).set_index("Model").round(4)

## 4. ROC Curve 비교

In [ ]:
fig = go.Figure()
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    auc = r["metrics"]["auc_roc"]
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} (AUC={auc:.4f})", mode="lines"))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name="Random", line=dict(dash="dash"), mode="lines"))
fig.update_layout(title="ROC Curve 비교", xaxis_title="FPR", yaxis_title="TPR")
fig.show()

## 5. Precision-Recall Curve 비교

In [ ]:
fig = go.Figure()
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r["y_proba"])
    fig.add_trace(go.Scatter(x=rec, y=prec, name=name, mode="lines"))
fig.update_layout(title="Precision-Recall Curve 비교", xaxis_title="Recall", yaxis_title="Precision")
fig.show()

## 6. Confusion Matrix 비교

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(results.keys()))
for i, (name, r) in enumerate(results.items(), 1):
    cm = confusion_matrix(y_test, r["y_pred"])
    fig.add_trace(
        go.Heatmap(z=cm, x=["유지", "이탈"], y=["유지", "이탈"],
                   text=cm, texttemplate="%{text}", colorscale="Blues", showscale=False),
        row=1, col=i
    )
fig.update_layout(title="Confusion Matrix 비교", height=400)
fig.show()

## 7. 피처 중요도 비교

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["XGBoost", "LightGBM"])
for i, name in enumerate(["XGBoost", "LightGBM"], 1):
    model = results[name]["model"]
    imp = model.feature_importances_
    feat_df = pd.DataFrame({"feature": all_features, "importance": imp})
    feat_df = feat_df.sort_values("importance", ascending=True).tail(10)
    fig.add_trace(
        go.Bar(x=feat_df["importance"], y=feat_df["feature"], orientation="h", name=name),
        row=1, col=i
    )
fig.update_layout(title="피처 중요도 비교 (Top 10)", height=500, showlegend=False)
fig.show()

## 8. 결론

- **XGBoost**가 AUC-ROC 기준 최고 성능
- 부스팅 모델이 Logistic Regression 대비 큰 폭으로 성능 향상
- 앙상블(Voting/Stacking)은 train.py에서 추가 실험 가능
- activity_score, SessionsPerWeek 등이 이탈 예측에 핵심 피처